# DEXTRA · 05 — Met-MoE: routing forecasts on the weather

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aduitya9/DEXTRA/blob/main/notebooks/05_met_moe.ipynb)

**Novelty 2.** Notebook 02 measured four of the 26 variates reversing their relationship with AQI between India's seasons — `blh_anomaly` is −0.17 in winter and +0.21 in the pre-monsoon; `fire_count_500km` −0.10 and +0.33. One forecasting head has to average those regimes into one mapping. Several heads, mixed by a router that knows which regime a day belongs to, need not.

### Four variants, three questions

| Variant | Head | Router | What it is |
|---|---|---|---|
| `backbone` | notebook 04's, target token only | — | the `w/o MoE` ablation row |
| `none` | one expert reading every variate's token | — | a control: isolates what a bigger head does on its own |
| `learned` | two such experts | the trunk's own encoding | the `w/o Met Gating` ablation row |
| `meteorological` | two such experts | the six weather gate features | **the full Met-MoE** |

1. **Does routing on physics beat routing on what the model learned?** — `meteorological` vs `learned`. This is the N2 claim.
2. **Does the mixture help at equal head capacity?** — `meteorological` vs `none`.
3. **Does the all-token head help on its own?** — `none` vs `backbone`.

The learned router is given *more* inputs and *more* parameters than the physics one, so if physics still wins it is not for want of either.

### Why every expert reads every variate
Built on a synthetic task whose answer flips sign with a regime flag, experts reading only the target's token stalled at exactly the "average the regimes away" error — nothing could start their specialisation. Experts reading all tokens separated the regimes almost perfectly on every seed. Under the *learned* router those same experts did not separate them at all: the trunk standardises each variate inside its window, so how strong an inversion is never reaches its encoding. The physics router reads it directly. Both results are now tests.

### Runtime
**GPU (T4).** Twelve runs of a minute or two each — under half an hour in all. On a CPU runtime the same work takes hours, so check the runtime type first.

In [ ]:
# Leave empty to read from the HF_DATASET_REPO Colab secret.
HF_DATASET_REPO = ""
GITHUB_REPO_URL = "https://github.com/Aduitya9/DEXTRA.git"

# The paper's two N2 ablation rows, the full model, and a control.
VARIANTS = ["backbone", "none", "learned", "meteorological"]
SEEDS = [42, 43, 44]          # three each, so a difference can be told from noise

# Architecture: notebook 04's settled values, so the trunk is identical everywhere.
D_MODEL, N_HEADS, N_LAYERS, DROPOUT = 128, 8, 2, 0.3

# Training
EPOCHS, BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, PATIENCE = 30, 256, 3e-4, 1e-3, 10
BALANCE_WEIGHT = 0.01         # config LOSS_WEIGHTS["lambda_balance"]

# Safety
TIME_BUDGET_HOURS = 10.5      # stop starting runs this far into the Colab session
CHECKPOINT_EVERY_MINUTES = 4  # a run longer than this saves its progress as it goes

SAVE_TO_HF = True

### If Colab disconnects
Free Colab cannot keep running once the tab is closed — background execution is a paid feature — and Colab's terms rule out scripts that fake activity to defeat the idle timeout, so this notebook does not use one. It is built so that a disconnect costs nothing instead:

- **Every finished run is saved to Hugging Face, with its weights, in one commit**, the moment it finishes.
- **A run still going when Colab drops saves its full training state** every few minutes — weights, optimiser, schedule, epoch, best-so-far and every random generator.
- **Re-running skips every finished run and resumes the interrupted one exactly where it stopped.** Same weights, same curve: a run killed after three epochs and resumed in a fresh session with a different seed is tested to finish on the identical validation curve.
- **Only the same run resumes.** Each checkpoint carries a signature of its settings, model shape and training data; change a setting or rebuild the data and the old checkpoint is set aside rather than quietly continued under the new settings.

So if you are disconnected: reconnect, **Run all**, and it carries on.

**Other safety measures in the loop:** early stopping on validation error with the *best* weights returned rather than the last; gradient clipping; a guard that ends a run on its best weights if the loss turns NaN or infinite instead of training on; a time budget measured from when the Colab VM booted, so its 12-hour ceiling can never cut a run off mid-save; and every Hugging Face read and write retried with backoff on a dropped connection or a server error, so a network blip cannot throw away a finished run.

**To make a disconnect unlikely in the first place:** keep this tab open, and stop your computer sleeping while it runs (Windows: *Settings → System → Power → Screen and sleep → Never*).

## Secrets
`HF_TOKEN` and `HF_DATASET_REPO`, as in the earlier notebooks.

In [ ]:
from google.colab import userdata

def colab_secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret '{name}' is missing or this notebook can't read it. Open the key icon "
            "(Secrets) in the left sidebar, add it, and switch on 'Notebook access'."
        )
    return value

HF_TOKEN = colab_secret("HF_TOKEN")
HF_DATASET_REPO = HF_DATASET_REPO or colab_secret("HF_DATASET_REPO")
print("Dataset repo:", HF_DATASET_REPO)

In [ ]:
%pip install -q "huggingface_hub>=0.23.0" pyarrow

In [ ]:
import io
import os
import subprocess
import sys
import time

REPO_DIR = "/content/DEXTRA"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", GITHUB_REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import torch
from src import config
from src.data import assemble
from src.evaluation import metrics
from src.models import itransformer, met_moe, sequences, training
from src.utils.hf_store import HFStore

device = itransformer.pick_device()
store = HFStore(HF_DATASET_REPO, token=HF_TOKEN)
daily = store.read_df("processed/city_daily_features.parquet")

# Recomputed from the current config rather than trusted as saved.
saved = daily["split"].copy() if "split" in daily.columns else None
daily["split"] = assemble.assign_split(daily)
differing = int((saved != daily["split"]).sum()) if saved is not None else 0
if differing:
    print(f"! {differing:,} rows differ from the saved splits; re-run notebook 02.")

# The 12-hour ceiling counts from when this VM booted, not from this cell.
try:
    session_age = float(open("/proc/uptime").read().split()[0])
except (OSError, ValueError, IndexError):
    session_age = 0.0
if not 0.0 <= session_age < 12 * 3600:   # not a Colab VM's age: count from now instead
    session_age = 0.0
deadline = time.monotonic() + max(0.0, TIME_BUDGET_HOURS * 3600 - session_age)

registry = training.RunRegistry(store, "runs/05")
planned = [f"{kind}_seed{seed}" for kind in VARIANTS for seed in SEEDS]
finished = [name for name in planned if registry.done(name)]

print(f"device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
if device.type != "cuda":
    print("! No GPU: this will take hours rather than minutes. "
          "Runtime -> Change runtime type -> T4 GPU, then Run all.")
print(f"{len(daily):,} city-days, {daily.city.nunique()} cities")
print(f"{len(finished)} of {len(planned)} runs already finished"
      + (" -- resuming where the last session stopped" if finished else ""))
print(f"time budget left this session: {(deadline - time.monotonic()) / 3600:.1f} h")

## A · Windows, and what the router sees
The windows are notebook 04's: 14 days of every variate in, three days of AQI out, cut inside each city, assigned to the split of their furthest target.

The physics router reads its six features **at their absolute value** — on day *t* and averaged over the fortnight — standardised with statistics from the training windows only. That is the difference from the trunk, which standardises each variate within its own window and so cannot tell a strong inversion from a weak one. `fire_count_500km` is logged first, or a single heavy fire day would set the router's whole scale.

In [ ]:
X, y, index = sequences.build_sequences(daily)
variates = sequences.feature_names(daily)
target_index = variates.index(sequences.TARGET)
is_train = (index.split == "train").to_numpy()
is_val = (index.split == "val").to_numpy()

gate_indices, gate_logged = met_moe.gate_layout(variates)
gate_centre, gate_scale = met_moe.gate_statistics(X[is_train], gate_indices, gate_logged)

print(f"X {X.shape}  y {y.shape}  ({is_train.sum():,} training windows)")
n_gate = len(gate_indices)
display(pd.DataFrame({
    "gate feature": config.GATE_FEATURES,
    "logged": gate_logged,
    "centre (day t)": gate_centre[:n_gate].round(3),
    "scale (day t)": gate_scale[:n_gate].round(3),
}))

## B · Train
Every variant, every seed, the same recipe. Finished runs are skipped; a run cut short by the time budget saves and stops, to resume next session. A run that diverges is *recorded* with its outcome rather than silently dropped, so it shows up in the results instead of vanishing from them.

In [ ]:
import tempfile
from pathlib import Path

ROUTED = ("learned", "meteorological")


def build(kind):
    common = dict(d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT)
    if kind == "backbone":
        return itransformer.ITransformer(X.shape[2], config.LOOKBACK_WINDOW,
                                         config.FORECAST_HORIZON, target_index, **common)
    return met_moe.MetMoE(X.shape[2], config.LOOKBACK_WINDOW, config.FORECAST_HORIZON,
                          target_index, gate_indices, gate_logged, router=kind,
                          gate_centre=gate_centre, gate_scale=gate_scale, **common)


def forecasts_of(model):
    predictions = itransformer.predict(model, X, device)
    return pd.concat([index.assign(horizon=h + 1,
                                   target_date=index["date"] + pd.Timedelta(days=h + 1),
                                   y=y[:, h], forecast=predictions[:, h])
                      for h in range(y.shape[1])], ignore_index=True)


def score_forecasts(long):
    out = {}
    for split in ("val", "test_seen", "test_unseen"):
        part = long[long.split == split]
        scored = metrics.score(part, "y", "forecast")
        severe = metrics.rare_event_metrics(part.y, part.forecast,
                                            threshold=config.RARE_AQI_THRESHOLD)
        out[split] = {"rmse": scored["rmse"], "mae": scored["mae"],
                      "recall": scored["recall"], "precision": scored["precision"],
                      "f1": scored["f1"], "support": scored["support"],
                      "severe_recall": severe["recall"], "severe_support": severe["support"]}
    return out


train_loader = itransformer.make_loader(X[is_train], y[is_train], BATCH_SIZE, shuffle=True)
val_loader = itransformer.make_loader(X[is_val], y[is_val], BATCH_SIZE * 2)

stopped = False
for kind in VARIANTS:
    for seed in SEEDS:
        name = f"{kind}_seed{seed}"
        if registry.done(name):
            print(f"\u2713 {name} already finished")
            continue
        if time.monotonic() > deadline:
            print("\u23f8 this session is near Colab's 12-hour limit -- "
                  "Runtime -> Disconnect and delete runtime, then Run all to continue")
            stopped = True
            break

        print(f"\n\u25b6 {name}")
        itransformer.set_seed(seed)
        model = build(kind)
        checkpointer = training.Checkpointer(store, f"checkpoints/05/{name}.pt",
                                             every_seconds=CHECKPOINT_EVERY_MINUTES * 60)
        started = time.monotonic()
        model, history, outcome = training.train(
            model, train_loader, val_loader, device, epochs=EPOCHS, lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY, patience=PATIENCE,
            aux_weight=BALANCE_WEIGHT if kind in ROUTED else 0.0,
            checkpointer=checkpointer, deadline=deadline, save_finished=False)
        if outcome == "out_of_time":
            print("\u23f8 saved partway -- Runtime -> Disconnect and delete runtime, "
                  "then Run all to resume this run")
            stopped = True
            break
        if outcome == "diverged":
            print("\u2717 diverged; recorded on its best weights so it stays visible")

        result = score_forecasts(forecasts_of(model))
        result.update({"variant": kind, "seed": seed, "outcome": outcome,
                       "parameters": itransformer.count_parameters(model),
                       "best_epoch": min(history, key=lambda h: h["val_rmse"])["epoch"] if history else 0,
                       "epochs_run": len(history), "history": history,
                       "minutes": round((time.monotonic() - started) / 60, 2)})
        if kind in ROUTED:
            result["regime"] = met_moe.regime_report(model.gate_weights(X, device), index, daily)
        if SAVE_TO_HF:
            with tempfile.TemporaryDirectory() as tmp:
                weights = Path(tmp) / f"{name}.pt"
                torch.save(model.state_dict(), weights)
                registry.record(name, result, attachments={f"{name}.pt": weights})
        unseen = result["test_unseen"]
        print(f"  \u2191 recorded -- unseen recall {unseen['recall']:.3f}, rmse {unseen['rmse']:.2f}"
              f", {result['minutes']} min")
    if stopped:
        break

finished = [name for name in planned if registry.done(name)]
print(f"\n{len(finished)} of {len(planned)} runs finished")

## C · Results, across seeds
Mean ± standard deviation over the three seeds. `recall` counts days above AQI 200; `severe_recall` days above 300, the band milestone M3 is defined on.

In [ ]:
results = [registry.load(name) for name in planned if registry.done(name)]
missing = [name for name in planned if not registry.done(name)]
if missing:
    print(f"! {len(missing)} runs not finished yet: {missing}")

rows = []
for result in results:
    for split in ("val", "test_seen", "test_unseen"):
        rows.append({"variant": result["variant"], "seed": result["seed"], "split": split,
                     **{k: result[split][k] for k in ("rmse", "recall", "f1", "severe_recall")}})
runs = pd.DataFrame(rows)

def mean_sd(series):
    return f"{series.mean():.3f} \u00b1 {series.std(ddof=1):.3f}" if len(series) > 1 else f"{series.mean():.3f}"

summary = (runs.groupby(["split", "variant"], sort=False)[["rmse", "recall", "f1", "severe_recall"]]
               .agg(mean_sd))
order = pd.MultiIndex.from_product([["val", "test_seen", "test_unseen"], VARIANTS])
display(summary.reindex([i for i in order if i in summary.index]))

epochs = pd.DataFrame([{k: r[k] for k in ("variant", "seed", "best_epoch", "epochs_run",
                                          "outcome", "parameters", "minutes")} for r in results])
display(epochs.groupby("variant", sort=False).agg(parameters=("parameters", "first"),
        best_epoch=("best_epoch", "mean"), minutes=("minutes", "mean"),
        outcomes=("outcome", lambda s: ", ".join(sorted(set(s))))).reindex(VARIANTS))

## D · The N2 verdict
Each question as a difference in means, with its standard error across seeds. With three seeds a difference counts as **clear** only when it exceeds twice that error; otherwise it is reported as within seed noise rather than claimed.

In [ ]:
questions = [("meteorological", "learned", "physics routing vs learned routing"),
             ("meteorological", "none", "mixture vs single expert, same head"),
             ("none", "backbone", "all-token head vs target-only head")]
better_when = {"rmse": "lower", "recall": "higher", "f1": "higher", "severe_recall": "higher"}

verdict = []
for a, b, question in questions:
    for split in ("test_seen", "test_unseen"):
        for metric, direction in better_when.items():
            A = runs[(runs.variant == a) & (runs.split == split)][metric].dropna()
            B = runs[(runs.variant == b) & (runs.split == split)][metric].dropna()
            if A.empty or B.empty:
                continue
            difference = A.mean() - B.mean()
            error = (np.sqrt(A.var(ddof=1) / len(A) + B.var(ddof=1) / len(B))
                     if len(A) > 1 and len(B) > 1 else np.nan)
            gain = -difference if direction == "lower" else difference
            call = ("within seed noise" if not np.isfinite(error) or abs(difference) <= 2 * error
                    else (f"{a} better" if gain > 0 else f"{b} better"))
            verdict.append({"question": question, "split": split, "metric": metric,
                            a: A.mean(), b: B.mean(), "difference": difference,
                            "\u00b1 error": error, "verdict": call})
verdict = pd.DataFrame(verdict)
for question in verdict.question.unique():
    print(f"\n{question}")
    part = verdict[verdict.question == question].dropna(axis=1, how="all")
    display(part.drop(columns="question").round(4))

## E · Did the gate find the physics?
For the expert that hazardous days lean on most: how its weight tracks the inversion and the night-time boundary layer, how it moves through the seasons, and how much more of it hazardous days receive. A router that learned the regimes should give that expert more weight on stagnant days — higher `inv_idx`, lower `blh_min` — and in the winter and post-monsoon smog seasons. The learned router is shown beside it for contrast.

In [ ]:
regimes = []
for result in results:
    if "regime" not in result:
        continue
    r = result["regime"]
    regimes.append({"variant": result["variant"], "seed": result["seed"],
                    "hazardous days": r["weight_on_hazardous_days"],
                    "other days": r["weight_on_other_days"],
                    "corr inv_idx": r["corr_with_inv_idx"],
                    "corr blh_min": r["corr_with_blh_min"],
                    **{f"{s}": v for s, v in r["by_season"].items()}})
regimes = pd.DataFrame(regimes)
if len(regimes):
    display(regimes.groupby("variant").mean(numeric_only=True).drop(columns="seed").round(3))
    print("Weights are for the expert hazardous days lean on most. Columns from winter on are")
    print("that expert's mean weight in each season.")

## F · Per city, and severe episodes against milestone M3
Milestone M3 asks the full DEXTRA model to beat the vanilla iTransformer by more than 25% on severe episodes. The full model adds C-STDA and the LoRA adapter, so this is a progress reading for N2 alone, not the milestone itself.

**Persistence is shown for scale** — tomorrow's AQI is today's. In the unseen cities severe days barely clear the line (they peak near 340), and a forecast trained on squared error flattens peaks just enough to fall short of 300, so persistence catches far more of them than any model here. The counts are shown too: on a base of a few hits, a relative gain like "+50%" can be two or three forecasts.

In [ ]:
severe = runs[runs.split.isin(["test_seen", "test_unseen"])].groupby(["split", "variant"])["severe_recall"].mean().unstack()
severe = severe.reindex(columns=[v for v in VARIANTS if v in severe.columns])
if "backbone" in severe.columns:
    for variant in severe.columns:
        if variant != "backbone":
            severe[f"{variant} vs backbone %"] = 100 * (severe[variant] / severe["backbone"] - 1)
for split in severe.index:
    part = (index.split == split).to_numpy()
    today = np.repeat(X[part, -1, target_index][:, None], y.shape[1], axis=1)
    reference = metrics.rare_event_metrics(y[part].ravel(), today.ravel(),
                                           threshold=config.RARE_AQI_THRESHOLD)
    severe.loc[split, "persistence"] = reference["recall"]
    severe.loc[split, "severe targets"] = reference["support"]
    severe.loc[split, "peak AQI"] = round(float(y[part].max()))
display(severe.round(3).astype({"severe targets": int, "peak AQI": int}))
print(f"M3 target: more than +25% on severe episodes, for the full model.")

## G · Save the Met-MoE
The meteorological run with the best validation error becomes `models/met_moe`. Its weights already sit in `runs/05/`, so the pointer names them rather than uploading a second copy. Its forecasts and per-city scores go up alongside, for notebooks 06 to 08.

In [ ]:
chosen = [r for r in results if r["variant"] == "meteorological"]
if not chosen:
    print("No meteorological run has finished yet.")
else:
    best = min(chosen, key=lambda r: r["val"]["rmse"])
    name = f"meteorological_seed{best['seed']}"
    model = build("meteorological")
    state = torch.load(io.BytesIO(store.read_bytes(registry.attachment(f"{name}.pt"))),
                       map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    model.to(device)
    long = forecasts_of(model)
    per_city = metrics.score_by(long[long.split == "test_unseen"], "city", "y", "forecast")
    display(per_city[["city", "n", "rmse", "support", "recall", "f1"]].round(3))
    if SAVE_TO_HF:
        spec = {"n_variates": int(X.shape[2]), "lookback": config.LOOKBACK_WINDOW,
                "horizon": config.FORECAST_HORIZON, "target_index": int(target_index),
                "gate_indices": gate_indices, "gate_logged": gate_logged,
                "router": "meteorological", "n_experts": 2, "d_model": D_MODEL,
                "n_heads": N_HEADS, "n_layers": N_LAYERS, "dropout": DROPOUT,
                "variates": variates, "gate_features": config.GATE_FEATURES,
                "weights": registry.attachment(f"{name}.pt"), "seed": best["seed"],
                "val_rmse": best["val"]["rmse"]}
        store.upload_json(spec, "models/met_moe.json", "Met-MoE: architecture and pointer to weights")
        keep = ["city", "date", "target_date", "horizon", "split", "y", "forecast"]
        store.upload_df(long[keep].rename(columns={"forecast": "met_moe"}),
                        "processed/met_moe_forecasts.parquet", "Met-MoE forecasts")
        store.upload_df(runs, "processed/met_moe_runs.csv", "Notebook 05: every run, every split")
        print(f"\u2191 models/met_moe.json  (weights: {spec['weights']})")
        print("\u2191 processed/met_moe_forecasts.parquet, processed/met_moe_runs.csv")
print("\nNext: notebook 06 adds C-STDA, the diffusion augmenter for rare episodes.")